# Guardrails & Output Validation — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/ai-safety/7-3/guardrails-implementation.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/ai-safety/7-3/guardrails-implementation.ipynb)

> Companion to the **Guardrails & Output Validation** article. Build a guardrail **wrapper** around a
> text generator and *watch* the core reliability pattern happen line by line:
> **generate → validate → retry (bounded) → fail visibly**.

## What you'll build

1. A **deterministic mock generator** that returns a **bad** output first, then a **good** one.
2. Two **output guardrails**: a **schema/format** check and a **grounding/overstatement** check.
3. A **wrapper** that runs the loop — generate, validate, retry up to `N`, and on final failure
   **stamp a visible `_error`** instead of shipping the raw unchecked text.
4. A **forced total failure** so you can see the fail-visibly branch fire.

**No API key needed.** Everything below runs on a scripted mock so you can watch the retry and the
pass deterministically. The **last section** shows the *real* Claude version — it runs in Colab once
you add your `ANTHROPIC_API_KEY`, and no-ops safely without one.

**The principle to watch for:** the LLM *proposes* a string; deterministic **code decides** whether
it ships.

### Setup (nothing to install for the mock sections)

In [ ]:
# Only needed for the FINAL "real Claude" section — safe to skip for now:
# %pip install -q anthropic
import os, json
print("ready — no API key required for the mock sections")

## The source of truth and the rules

The generator must answer from ONE approved context. The guardrails encode what a good answer
looks like: the required JSON shape, and language that must NOT appear (absolutes the source
doesn't support). This is the ground truth the deterministic checks measure against.

In [ ]:
CONTEXT = ("In a 2-year trial the treatment lowered heart-attack risk from 30% to 15%. "
           "It has not been tested in patients under 18.")

REQUIRED_KEYS = ("summary", "risk_reduction", "caveat")          # schema/format rule
BANNED_WORDS  = ("eliminates", "cures", "guarantees", "100%",    # overstatement / ungrounded
                 "safe for everyone", "all ages")

print("context:", CONTEXT)
print("required keys:", REQUIRED_KEYS)
print("banned (overstated) phrases:", BANNED_WORDS)

## The mock generator — bad first, then good

A stand-in for the LLM. It ignores its inputs and simply returns the scripted output for each
attempt: attempt 0 is **malformed** (a key is missing), attempt 1 is **grounded and well-formed**.
Real LLMs emit the same shape — a string we must validate before trusting.

In [ ]:
_SCRIPT = [
    # attempt 0 — BAD: missing the required "caveat" key AND overstated language
    '{"summary": "The drug eliminates heart attacks and is safe for everyone.", '
        '"risk_reduction": "cuts risk by half"}',
    # attempt 1 — GOOD: correct schema, grounded, hedged to the source
    '{"summary": "Over 2 years the drug lowered heart-attack risk from 30% to 15%.", '
        '"risk_reduction": "about half", "caveat": "not tested in patients under 18"}',
]

def mock_generate(context, feedback, attempt):
    """Stand-in for an LLM. Returns the scripted output for this attempt."""
    return _SCRIPT[min(attempt, len(_SCRIPT) - 1)]

print("attempt 0 ->", mock_generate(CONTEXT, "", 0)[:60], "...")
print("attempt 1 ->", mock_generate(CONTEXT, "", 1)[:60], "...")

## The two guardrails — deterministic CODE that DECIDES

`check_schema` enforces structure (valid JSON, all keys present, all strings). `check_grounding`
enforces content (no overstated/ungrounded language). Each returns **pass/fail + a reason** — the
reason is what we feed back on retry. Neither is an LLM: they are plain, testable code.

In [ ]:
def check_schema(text):
    """Gate 1 — structure. Valid JSON with all required string fields?"""
    try:
        obj = json.loads(text)
    except json.JSONDecodeError as e:
        return False, f"not valid JSON ({e})", None
    missing = [k for k in REQUIRED_KEYS if k not in obj]
    if missing:
        return False, f"missing keys: {missing}", None
    if not all(isinstance(obj[k], str) for k in REQUIRED_KEYS):
        return False, "all fields must be strings", None
    return True, "schema ok", obj

def check_grounding(obj):
    """Gate 2 — grounding / overstatement. Any absolute claim the source doesn't support?"""
    blob = " ".join(obj[k].lower() for k in REQUIRED_KEYS)
    hits = [w for w in BANNED_WORDS if w in blob]
    if hits:
        return False, f"overstated / ungrounded language: {hits}"
    return True, "grounded ok"

# quick smoke test on the bad attempt-0 string
ok, reason, obj = check_schema(mock_generate(CONTEXT, "", 0))
print("schema on attempt 0 ->", ok, "|", reason)

## The wrapper — generate → validate → retry (bounded) → fail visibly

The whole pattern in one function. Watch each attempt print: what the model **proposed**, which
gate **failed** (and why), or that all gates **passed**. Retries are **bounded** by `max_retries`.
On success it returns the trusted object; on exhausting the budget it returns a visible `_error`
and **never** the raw unchecked text.

In [ ]:
def guarded_generate(context, max_retries=2, verbose=True):
    feedback = ""
    for attempt in range(max_retries + 1):        # attempt 0 is the first try
        text = mock_generate(context, feedback, attempt)
        if verbose:
            print(f"attempt {attempt}: model proposed -> {text[:60]}...")

        ok, reason, obj = check_schema(text)              # gate 1: structure
        if not ok:
            if verbose: print(f"           schema FAIL -> {reason}  (retry)")
            feedback = reason
            continue
        ok2, reason2 = check_grounding(obj)               # gate 2: grounding/overstatement
        if not ok2:
            if verbose: print(f"           grounding FAIL -> {reason2}  (retry)")
            feedback = reason2
            continue

        if verbose: print("           ALL CHECKS PASS -> trusted output")
        return obj                                        # only a fully-validated output ships

    # bounded retries exhausted: FAIL VISIBLY, never return the raw unchecked text
    return {"_error": f"guardrail failed after {max_retries} retries: {feedback}",
            "_attempts": max_retries + 1}

result = guarded_generate(CONTEXT, max_retries=2)
print("\nFINAL (trusted):", json.dumps(result, indent=2))

## Force a total failure — watch it FAIL VISIBLY

Now make **every** attempt bad and shrink the budget. The wrapper exhausts its retries and returns
an `_error` object. Notice the raw, malformed, overstated text is **never** returned — a bad output
is quarantined behind a visible flag, exactly as a safe system requires.

In [ ]:
# monkeypatch the generator so every attempt returns the BAD output
_orig_script = _SCRIPT.copy()
_SCRIPT[:] = [_orig_script[0], _orig_script[0]]   # bad, then bad again

failed = guarded_generate(CONTEXT, max_retries=1)
print("\nFINAL (failed):", json.dumps(failed, indent=2))
print("\nraw text shipped? ->", "NO — only an _error object" if "_error" in failed else "YES (bug!)")

_SCRIPT[:] = _orig_script   # restore the good script

## Watch the difference

| Outcome | What the wrapper returns | Safe? |
|---|---|---|
| a later attempt passes every gate | the **trusted object** | yes — validated before shipping |
| every attempt fails within budget | an **`_error` object** | yes — bad text quarantined, failure visible |
| (the unsafe anti-pattern) | the **raw unchecked text** | NO — a bad output looks trusted |

**The one interview test:** on final failure, does a raw model string ever reach the caller?
If yes, you have a silent-fail bug. If no — only a trusted object or a visible error — the
guardrail is doing its job.

## The real thing (Claude) — run this in **Colab** with your key

Identical guardrail loop; the mock generator is swapped for a real `client.messages.create` call.
It runs when `ANTHROPIC_API_KEY` is set (Colab → *Secrets*). With no key it prints a friendly note,
so the notebook still executes top-to-bottom. The validators — the deterministic **code that
decides** — are unchanged. Only the *proposer* changes.

In [ ]:
def claude_generate(context, feedback, attempt):
    """Real generator: ask Claude for grounded JSON, feeding back the last failure reason."""
    from anthropic import Anthropic
    client = Anthropic()
    instructions = (
        "Answer ONLY from the context. Return JSON with keys: summary, risk_reduction, caveat. "
        "Do not overstate — no absolute claims the context does not support."
    )
    if feedback:
        instructions += f" Your previous attempt was rejected: {feedback}. Fix it."
    resp = client.messages.create(
        model="claude-sonnet-4-5", max_tokens=512,
        system=instructions,
        messages=[{"role": "user", "content": f"CONTEXT: {context}\n\nSummarize for a patient as JSON."}],
    )
    return resp.content[0].text

def guarded_generate_claude(context, max_retries=2):
    if not os.getenv("ANTHROPIC_API_KEY"):
        print("(no ANTHROPIC_API_KEY set — add it in Colab Secrets to run the real Claude loop)")
        return None
    feedback = ""
    for attempt in range(max_retries + 1):
        text = claude_generate(context, feedback, attempt)
        print(f"attempt {attempt}: Claude proposed -> {text[:60]}...")
        ok, reason, obj = check_schema(text)
        if not ok:
            print(f"           schema FAIL -> {reason}  (retry)"); feedback = reason; continue
        ok2, reason2 = check_grounding(obj)
        if not ok2:
            print(f"           grounding FAIL -> {reason2}  (retry)"); feedback = reason2; continue
        print("           ALL CHECKS PASS -> trusted output")
        return obj
    return {"_error": f"guardrail failed after {max_retries} retries: {feedback}",
            "_attempts": max_retries + 1}

print(guarded_generate_claude(CONTEXT))

## Recap

- A **guardrail** wraps the generator; the LLM **proposes**, deterministic **code decides**.
- The pattern is **generate → validate → retry (bounded) → fail visibly** — cap retries so you
  can't loop forever.
- Two gates ran here: a **schema/format** check and a **grounding/overstatement** check. An output
  ships only after clearing **every** gate.
- On final failure the wrapper returns a visible **`_error`** and **never** the raw unchecked text —
  the difference between a safe system and a dangerous one.

← Back to the **Guardrails & Output Validation** article for the full explanation, diagrams, and interview Q&A.